# MTL 3채널 (sp + bondpp + frbsf_wr) — 변종 A ablation

**검증 목적**: ★ best (sp + bondpp 2ch, sp_return median = −2.199) 위에 `frbsf_wr` 를 target 채널로 추가했을 때 학습이 향상/유지/악화되는지.

**근거**: `frbsf_wr` 직교성 통과 — VIF 1.044 (full 기준 10개 후보 중 1위), max|r| 0.158 (sp_return), weekly nz_diff_pct 100% (forward-fill 문제 없음).

**실험 구성** (2 변종 × 5 시드 = 10 run):
1. MTL 3ch (sp + bondpp + frbsf_wr), bondpp 정규화 X
2. MTL 3ch (sp + bondpp + frbsf_wr), bondpp 정규화 O

**비교 대상**:
- ★ best (sp + bondpp 2ch): sp_return median = −2.199, std 0.075
- MTL 3ch (sp + liq + bondpp): sp_return median = −2.16 (채널 추가 시 negative transfer)

**준비**: T4 또는 A100 런타임 → 셀 순차 실행. 결과는 셀 출력에 그대로 표시됨 (.log 파일 저장 X).

## 1. Drive mount + repo clone/pull + cd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Colab Notebooks', exist_ok=True)
os.chdir('/content/drive/MyDrive/Colab Notebooks')

if not os.path.exists('homeostatic-market'):
    !git clone https://github.com/hwayobi2020/homeostatic-market.git
    print('Cloned fresh.')
else:
    os.chdir('/content/drive/MyDrive/Colab Notebooks/homeostatic-market')
    !git pull
    print('Pulled latest.')

os.chdir('/content/drive/MyDrive/Colab Notebooks/homeostatic-market/colab/dual_3ch')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

## 2. GPU 확인

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
    print('VRAM total     :', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 3. 데이터 빌드 — FRBSF weekly + weekly_ppbond 병합

`data/news_sentiment_frbsf_latest.xlsx` (일별 1980-01~2026-03) → `colab/dual_3ch/data/weekly_ppbond_frbsf_{train,test}.csv` (33 컬럼, 기존 32 + `frbsf_wr`).

이미 commit 되어 있다면 스킵 가능. 새로 빌드해도 1초.

In [ ]:
!python ../../data/build_frbsf_weekly.py

## 4. (선택) 직교성 점검 재실행

VIF + Pearson |r| + train/test 일관성. `frbsf_wr` 가 1위 (VIF 1.04) 임을 재확인.

In [ ]:
!python ../../analysis/orthogonality_check.py

## 5. 변종 1 — MTL 3ch (sp + bondpp + frbsf_wr), bondpp 정규화 X (5 시드)

T4 약 3분, A100 약 1.5분. 매 epoch test NLL 이 셀 출력에 실시간 표시.

In [ ]:
!python -u train_mtl_bondpp_frbsf.py --seeds 42 123 777 0 99

## 6. 변종 2 — MTL 3ch (sp + bondpp + frbsf_wr), bondpp 정규화 O (5 시드)

`pp_bond_13w_lag` 만 train mean/std 로 standardize. `sp_return` 과 `frbsf_wr` 는 raw 유지.

In [ ]:
!python -u train_mtl_bondpp_frbsf.py --seeds 42 123 777 0 99 --normalize-bondpp

## 7. 결과 비교 — 신규 변종 vs ★ best vs MTL 3ch (sp+liq+bondpp)

`test_sp_return` median 을 −2.199 (★ best) 와 비교. 음수 방향으로 더 작으면 향상, 더 크면 악화.

In [ ]:
import pandas as pd
import os

BASELINES = {
    '★ best (sp + bondpp 2ch)':                 (-2.199, 0.075),
    'MTL 3ch (sp + liq + bondpp)':              (-2.16,  0.09),
    'MTL 3ch (sp + liq + vix)':                 (-2.21,  0.10),
}

rows = []
for tag, label in [('mtl_bp_frbsf',         'MTL 3ch (sp + bondpp + frbsf_wr) raw'),
                   ('mtl_bp_frbsf_normbp',  'MTL 3ch (sp + bondpp + frbsf_wr) normbp')]:
    p = f'result/{tag}_multiseed_results.csv'
    if not os.path.exists(p):
        print(f'(skip) {p} 없음 — 학습 미완')
        continue
    df = pd.read_csv(p)
    print(f'\n=== {label} ===')
    print(df.to_string(index=False))
    sp_med  = df['test_sp_return'].median()
    sp_mean = df['test_sp_return'].mean()
    sp_std  = df['test_sp_return'].std()
    rows.append((label, sp_med, sp_mean, sp_std, len(df)))

print('\n' + '#' * 80)
print('# 비교 표 — sp_return test NLL')
print('#' * 80)
print(f"  {'모델':<45s} {'median':>9s} {'mean':>9s} {'std':>8s} {'n':>4s}")
print('  ' + '-' * 78)
for label, (med, std) in BASELINES.items():
    print(f'  {label:<45s} {med:>+9.4f} {"-":>9s} {std:>8.4f} {"-":>4s}')
for label, med, mean, std, n in rows:
    diff = med - (-2.199)
    arrow = '↓' if diff < 0 else ('↑' if diff > 0 else '=')
    print(f'  {label:<45s} {med:>+9.4f} {mean:>+9.4f} {std:>8.4f} {n:>4d}   ({arrow}{abs(diff):.4f} vs ★)')